# Instructor Notebook 01 — Machine Learning
**ComplianceGPT Lab · REU 2026 · Week 2**

> **Teaching script:** Run every cell top-to-bottom. All outputs are deterministic (random_state fixed). Use markdown cells as talking points.

**Learning arc:**
1. What ML actually is (data replaces rules)
2. Build intuition with FIFA 2014 — accessible, fun, numeric
3. Show the three algorithms, compare them
4. Pivot to HIPAA — expose the feature extraction problem
5. Land the motivation for NLP (next session)

---
## Part 1 · The Core Idea

> **Say:** "Before ML, you wrote rules by hand. `if 'court order' in text: return PERMITTED`. The problem? 200 rules, every edge case breaks something, and 'judicial mandate' slips through. ML inverts this: you show the algorithm labeled examples, it finds the rules itself."

The three ingredients:
- **Features** — numeric representation of your input
- **Labels** — the correct answer for each example
- **Algorithm** — finds the pattern from (features, labels) pairs

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

---
## Part 2 · FIFA 2014 — Build Intuition

> **Say:** "Let's use FIFA 2014. 32 teams. We know who won each match. Can an algorithm predict which teams made the top 8 — the quarterfinals — just from their stats?"

> **Say:** "This data is clean, numeric, and everyone understands it. The same algorithm you learn here runs on HIPAA scenarios."

In [ ]:
FIFA_PATH = '/Users/priscilladanso/Downloads/FIFA - 2014.csv'

fifa = pd.read_csv(FIFA_PATH)

# Goal Difference uses Unicode minus sign — fix it
fifa['Goal Difference'] = pd.to_numeric(
    fifa['Goal Difference'].astype(str).str.replace('\u2212', '-'), errors='coerce'
)

print('FIFA 2014 World Cup — All 32 teams')
print(f'Shape: {fifa.shape}')
print()
print(fifa.to_string(index=False))

### What are we predicting?

> **Say:** "We want to predict: did this team reach the **quarterfinals** (top 8)? That means they played 5+ games."

> **Say:** "This is a **binary classification** problem. Two possible answers: yes (top 8) or no (knocked out early)."

In [ ]:
# Label: reached the quarterfinals (played 5+ games)
fifa['top8'] = (fifa['Games Played'] >= 5).astype(int)

print('Label distribution:')
print(fifa['top8'].value_counts().rename({1:'Reached QF (top 8)', 0:'Knocked out early'}).to_string())
print()
print('Teams that reached the quarterfinals:')
print(fifa[fifa['top8']==1][['Position','Team','Games Played','Win','Goals For']].to_string(index=False))

### Choose Features

> **Say:** "Features are what we feed the algorithm. We don't include `Games Played` — that would be cheating, it's almost directly our label. We use stats the team accumulated: wins, losses, goals, goal difference."

> **Ask students:** "Which feature do you think will matter most?"

In [ ]:
FEATURES = ['Win', 'Draw', 'Loss', 'Goals For', 'Goals Against', 'Goal Difference', 'Points']

X = fifa[FEATURES].values
y = fifa['top8'].values

print('Feature matrix X shape:', X.shape, '  →  32 teams × 7 features')
print('Label vector y shape:  ', y.shape, '  →  0 or 1 for each team')
print()
print('First 5 rows of X (Wins, Draws, Losses, GF, GA, GD, Points):')
print(pd.DataFrame(X[:5], columns=FEATURES, index=fifa['Team'][:5]).to_string())

---
## Part 3 · Three Algorithms

> **Say:** "We're going to try three classifiers. Same features, same labels — the algorithm is the only thing that changes. This lets us compare their approaches directly."

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

print(f'{'Model':<25} {'CV Accuracy':>12} {'Std':>8}')
print('-' * 50)
results = {}
for name, model in models.items():
    X_in = X_scaled if name == 'Logistic Regression' else X
    scores = cross_val_score(model, X_in, y, cv=5, scoring='accuracy')
    results[name] = scores
    print(f'{name:<25} {scores.mean():>12.1%} {scores.std():>8.3f}')

print()
print('Note: 32 samples = small dataset. CV variance is high. Use these as rough comparisons.')

### Decision Tree — Read the Rules

> **Say:** "The Decision Tree is special: you can read exactly what it learned. Let's see the rules it found."

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X, y)

print('Decision Tree rules (depth=3):')
print('='*50)
print(export_text(dt, feature_names=FEATURES))
print('class labels: 0=knocked out  1=reached QF')

### Logistic Regression — Feature Weights

> **Say:** "Logistic Regression assigns a weight to each feature. Positive weight = pushes toward 'top 8'. Negative weight = pushes toward 'knocked out'. Look at which features matter most."

In [ ]:
lr = LogisticRegression(random_state=42)
lr.fit(X_scaled, y)

weights = pd.Series(lr.coef_[0], index=FEATURES).sort_values(ascending=False)
print('Logistic Regression feature weights (scaled):')
print('Positive = pushes toward top-8   Negative = pushes toward early exit')
print()
for feat, w in weights.items():
    bar = '█' * int(abs(w) * 3)
    sign = '+' if w > 0 else '-'
    print(f'  {feat:<20} {sign}{abs(w):.2f}  {bar}')

### Random Forest — Feature Importances

> **Say:** "Random Forest trains 100 trees, each on a random sample of data and features. Their vote determines the final prediction. The feature importance shows how often each feature was useful for splitting."

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Random Forest — Feature Importances')
print()
for feat, imp in importances.items():
    bar = '█' * int(imp * 50)
    print(f'  {feat:<20} {imp:.3f}  {bar}')

### Predict on a New Team

> **Say:** "Now let's create a hypothetical team and see what the models predict. This is what prediction looks like in practice."

In [ ]:
# Hypothetical team stats
new_team = pd.DataFrame([{
    'Win': 3, 'Draw': 1, 'Loss': 0,
    'Goals For': 9, 'Goals Against': 2,
    'Goal Difference': 7, 'Points': 10
}])

print('Hypothetical team: 3W 1D 0L, 9 goals scored, 2 conceded')
print()

for name, (model, X_in) in {
    'Logistic Regression': (lr, scaler.transform(new_team.values)),
    'Decision Tree':       (dt, new_team.values),
    'Random Forest':       (rf, new_team.values),
}.items():
    pred = model.predict(X_in)[0]
    prob = model.predict_proba(X_in)[0][1]
    label = 'REACHED QF ✓' if pred == 1 else 'KNOCKED OUT ✗'
    print(f'  {name:<25} → {label}  (confidence: {prob:.0%})')

---
## Part 4 · The Wall — Pivot to HIPAA

> **Say:** "FIFA was easy. The features were already numbers — wins, goals, points. What if your input is a paragraph of legal text? You can't feed that to sklearn. You need to convert text into numbers first. That's the feature extraction problem."

> **Say:** "Here's what the HIPAA data actually looks like. Our model already ran on it — using a Language Model to extract boolean features. Let's see what it produced."

In [ ]:
HIPAA_PATH = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv'

hipaa = pd.read_csv(HIPAA_PATH)
print(f'HIPAA GoldCoin benchmark: {len(hipaa)} court cases')
print(f'Correct predictions: {(hipaa["match"]=="Y").sum()} / {len(hipaa)} = {(hipaa["match"]=="Y").mean():.1%}')
print()
print('Example scenario (row 0):')
print('-'*60)
print(hipaa['question'].iloc[0][:400], '...')
print()
print('Ground truth:', hipaa['ground_truth'].iloc[0])
print('Model verdict:', hipaa['verdict_norm'].iloc[0])

In [ ]:
import json

# Show what the LLM extracted from the same scenario
sj = hipaa['scenario_json'].iloc[0]
try:
    parsed = json.loads(sj) if isinstance(sj, str) else sj
    print('What the LLM extracted from that text:')
    print('-'*60)
    for k, v in parsed.items():
        print(f'  {k:<40} : {v}')
except Exception as e:
    print('scenario_json:', str(sj)[:600])

### The Key Point

> **Say:** "Notice what just happened. The LLM read 200 words of legal text and extracted ~30 boolean features automatically. *Then* a rules engine (Soufflé) ran on those features. The ML we did on FIFA — that's exactly the logic inside the rules engine."

> **Say:** "The hard part isn't the decision logic. The hard part is getting from raw text to clean features. That's what NLP and LLMs solve. That's what this week is about."

**The ML wall you just hit:**
- Raw HIPAA text → sklearn → ❌ can't do this directly
- Raw HIPAA text → LLM extraction → boolean features → rules engine → ✓
- Question: can we skip the LLM and go directly from text to verdict?
- Answer (tomorrow): kind of, but it's worse. Here's why...

In [ ]:
# Teaser: what if we tried to use the raw text directly?
# (We'll do this properly in teach_03_nlp.ipynb)
from sklearn.feature_extraction.text import TfidfVectorizer

texts = hipaa['question'].fillna('').tolist()
labels = (hipaa['ground_truth'] == 'PERMITTED').astype(int).tolist()

tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_text = tfidf.fit_transform(texts)

lr_text = LogisticRegression(random_state=42)
scores = cross_val_score(lr_text, X_text, labels, cv=5, scoring='accuracy')

print('Raw text → TF-IDF features → Logistic Regression')
print(f'Accuracy: {scores.mean():.1%} ± {scores.std():.3f}')
print()
print('Compare to LLM extraction → rules engine:')
print(f'Accuracy: {(hipaa["match"]=="Y").mean():.1%}')
print()
print('→ The gap shows why text representation matters.')
print('→ Tomorrow: why TF-IDF is limited, and what embeddings fix.')

---
## Summary

| Concept | What it means |
|---|---|
| **Features** | Numeric representation of your input |
| **Labels** | The correct answer for each example |
| **Logistic Regression** | Weighted sum → sigmoid → probability. Interpretable weights. |
| **Decision Tree** | Sequence of if/else splits. Fully readable. |
| **Random Forest** | 100 trees, majority vote. Best accuracy. |
| **The wall** | ML needs numeric features. Text → numbers = NLP problem. |

> **Transition:** "Tomorrow: Bag of Words, TF-IDF, and embeddings — three ways to turn text into numbers. Each one gets us closer to understanding HIPAA compliance."